# Кандидатогенерация для поиска услуг Авито

Задача: для каждого запроса из бенчмарка вернуть до 50 объявлений так, чтобы выбранные
пользователем попали в этот список. Метрика — **Recall@50**, среднее по запросам.

Ноутбук проходит весь путь решения: валидация → текст → гео → пул кандидатов → ранкер → сабмит.
В конце сверяет полученный `answer.csv` с отправленным по контрольной сумме.

Время полного прогона — около 10 минут на 16 ядрах, 12 ГБ памяти в пике.
Данные (`train.parquet`, `benchmark_items.parquet`, `benchmark_queries.parquet`) должны лежать в `data/`.

In [1]:
import hashlib, json, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PY = sys.executable
SEED = 42

from src import data as D
from src.bm25 import BM25FBuilder, single_field_index
from src.geo import haversine_km, location_centers
from src.metrics import topk_from_scores
from src.params import parse_many
from src.paths import val_dir
from src.text import Lemmatizer

def run(*args):
    "Запуск шага пайплайна тем же интерпретатором; вывод показываем как есть."
    t = time.time()
    r = subprocess.run([PY, *args], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout[-2000:] or r.stderr[-2000:])
    if r.returncode:
        raise RuntimeError(f"{args} завершился с кодом {r.returncode}")
    print(f"[{time.time()-t:.0f} c]")

for f in ("train.parquet", "benchmark_items.parquet", "benchmark_queries.parquet"):
    assert (ROOT / "data" / f).exists(), f"нет data/{f}"
print("данные на месте")

данные на месте


## 1. Локальная валидация

Судить качество по лидерборду нельзя — попыток мало. Нужна локальная валидация, повторяющая
устройство бенчмарка.

**Событие** = строки train с одинаковыми (нормализованный запрос, локация, фильтры); позитивы
события = выбранные в них объявления. Берём 3000 событий, из них:

* 62.5% **unseen** — запрос удаляется из обучающей части целиком;
* 37.5% **seen** — удаляется одно событие, остальные события запроса остаются.

Доли не выдуманы: в бенчмарке 37.5% запросов встречаются в train. Без такого разделения любая
«история по запросу» покажет рост, которого на тесте не будет.

Корпус валидации = корпус бенчмарка + позитивы валидации. Проверяем, что доля объявлений,
знакомых обучающей части, получилась как на бенчмарке (там 9.6%).

In [2]:
run("scripts/build_val.py", "--seed", str(SEED))

{
  "seed": 42,
  "fit_rows": 485437,
  "dropped_rows": 12236,
  "val_events": 3000,
  "seen_share": 0.375,
  "mean_positives": 1.128,
  "positives_unique": 3367,
  "corpus_size": 192344,
  "corpus_seen_in_fit_share": 0.0989,
  "positives_in_corpus": 3367,
  "seconds": 5.9
}

[6 c]


## 2. Точка отсчёта: BM25 + гео

Считаем BM25 по слитному полю «заголовок + сырые параметры» на леммах и сравниваем гео-режимы.

Гео нужно потому, что услуги локальны: в 83% пар train локация поиска совпадает с локацией
объявления, медианное расстояние до выбранного объявления 3.9 км. Но жёсткий фильтр по локации
запрещён — у 17% запросов бенчмарка в своей локации нет ни одного объявления. Поэтому не фильтр,
а **приоритет**: объявления внутри радиуса идут первыми, остальные не выбрасываются.

Метрики: R@50 (целевая), R@200 и R@1000 (сколько позитивов вообще попадает в пул) и разбивка
R@50 на seen/unseen.

In [3]:
run("scripts/eval_bm25_baseline.py", "--seed", str(SEED))
pd.read_csv(ROOT / f"results/baseline_bm25_seed{SEED}/summary.csv", index_col=0).round(4)

лемматизация: 7 с, уникальных токенов 604048
индекс: 92119 терминов x 192344 документов, 20 с
валидационных запросов без центра локации: 0.0000
  0/3000 событий, 22 с
  1000/3000 событий, 30 с
  2000/3000 событий, 38 с

                  R@50   R@200  R@1000  R@50_seen  R@50_unseen
variant
global          0.3141  0.4742  0.6849     0.3127       0.3150
same_loc_first  0.6847  0.7784  0.8778     0.6977       0.6769
r30km           0.7048  0.7865  0.8721     0.7180       0.6969
r80km           0.6903  0.7872  0.8693     0.7156       0.6751
r200km          0.6446  0.7652  0.8443     0.6666       0.6314

готово за 43 с -> results/baseline_bm25_seed42

[44 c]


,R@50,R@200,R@1000,R@50_seen,R@50_unseen
variant,,,,,
global,0.3141,0.4742,0.6849,0.3127,0.3150
same_loc_first,0.6847,0.7784,0.8778,0.6977,0.6769
r30km,0.7048,0.7865,0.8721,0.7180,0.6969
r80km,0.6903,0.7872,0.8693,0.7156,0.6751
r200km,0.6446,0.7652,0.8443,0.6666,0.6314


Радиус 30 км оказывается лучше 80 км на 1.5 пункта — то есть окрестность стоит брать плотнее,
чем кажется. R@1000 при этом почти не меняется: радиус влияет на порядок, а не на состав пула.

## 3. Поля документа и BM25F

Параметры объявления (`item_infm_params_text`) лежат плоской строкой «Ключ Значение Ключ Значение»
без разделителей, и почти вся она — шаблон, одинаковый у всех: «Начальная цена», «График работы
от 28800», «Время работы, с 08:00», «Дни пн Дни вт». Полезного там три вещи: вид и тип услуги,
названия услуг из прайса, место оказания.

Ключи — закрытый словарь, его добывает `scripts/discover_param_keys.py`. Разбор в `src/params.py`
делит строку на поля `kind` / `service` / `place` / `other`, шаблонные пары выбрасывает.

Дальше BM25F: поля с разными весами и разной нормировкой длины (заголовок короткий и точный —
нормируем слабо, описание длинное и рыхлое — сильно). Считаем вклад каждого шага отдельно.

In [4]:
# Строим индекс один раз и переиспользуем его для всех сравнений ниже.
vq = pd.read_parquet(val_dir(SEED) / "val_queries.parquet")
vc = pd.read_parquet(val_dir(SEED) / "val_corpus.parquet")
fit_mask = np.load(val_dir(SEED) / "fit_mask.npy")

lm = Lemmatizer()
parsed = parse_many(vc.item_infm_params_text.fillna("").tolist())
fields = {
    "title":   lm.many(vc.item_title_raw.fillna("").tolist()),
    "kind":    lm.many(parsed["kind"]),
    "service": lm.many(parsed["service"]),
    "place":   lm.many(parsed["place"]),
    "other":   lm.many(parsed["other"]),
    "desc":    lm.many(vc.item_description_raw.fillna("").tolist()),
}
params_raw = lm.many(vc.item_infm_params_text.fillna("").tolist())
qs = lm.many(vq.search_query.tolist())
lm.save()

B = {"title": 0.3, "kind": 0.3, "service": 0.75, "place": 0.5, "other": 0.75, "desc": 0.9, "params_raw": 0.75}
print(f"корпус {len(vc)}, запросов {len(vq)}, словарь лемм {lm.vocab_size}")

корпус 192344, запросов 3000, словарь лемм 604048


In [5]:
# Гео считаем один раз: центры локаций — медиана координат выбранных объявлений по обучающей части.
g = D.read_train(columns=["search_location_id", "item_latitude", "item_longitude"])[fit_mask]
qc = location_centers(g).reindex(vq.search_location_id)
qlat, qlon = np.radians(qc.lat.to_numpy()), np.radians(qc.lon.to_numpy())
ilat, ilon = np.radians(vc.item_latitude.to_numpy()), np.radians(vc.item_longitude.to_numpy())
ids = vc.item_id.to_numpy()
positives = [set(p) for p in vq.pos]
seen = vq.seen.to_numpy()

dist = np.empty((len(vq), len(vc)), dtype=np.float32)
for i in range(len(vq)):
    dist[i] = 1e4 if np.isnan(qlat[i]) else haversine_km(qlat[i], qlon[i], ilat, ilon)
log_dist = np.log1p(dist)
print(f"матрица расстояний {dist.shape}, {dist.nbytes/1e9:.1f} ГБ")

матрица расстояний (3000, 192344), 2.3 ГБ


In [6]:
KS = (50, 200, 1000)

def recall(W, Q, geo, batch=200):
    "R@K по всем событиям для матрицы весов W и функции гео-поправки geo(срез)."
    hits = {k: np.zeros(len(vq)) for k in KS}
    for s in range(0, len(vq), batch):
        e = min(s + batch, len(vq))
        S = (Q[s:e] @ W).toarray() + geo(slice(s, e))
        for j in range(e - s):
            i = s + j
            top = topk_from_scores(S[j], ids, max(KS))
            for k in KS:
                hits[k][i] = len(positives[i] & set(top[:k])) / len(positives[i])
    r = {f"R@{k}": hits[k].mean() for k in KS}
    r["R@50 seen"], r["R@50 unseen"] = hits[50][seen].mean(), hits[50][~seen].mean()
    return r

hard30 = lambda sl: 1000.0 * (dist[sl] < 30)
rows = {}

# а) заголовок + сырые параметры одним полем — как в точке отсчёта
b = BM25FBuilder().fit({"title": fields["title"], "params_raw": params_raw}, b=B)
rows["заголовок + сырые параметры"] = recall(b.combine({"title": 3, "params_raw": 1}), b.transform_queries(qs), hard30)

# б) разобранные поля, описание не используется
b = BM25FBuilder().fit({k: v for k, v in fields.items() if k != "desc"}, b=B)
w = {"title": 3, "kind": 2, "service": 2, "place": 0.5, "other": 0.3}
rows["BM25F по полям, без описания"] = recall(b.combine(w), b.transform_queries(qs), hard30)

# в) плюс описание
builder = BM25FBuilder().fit(fields, b=B)
Q = builder.transform_queries(qs)
W = builder.combine({**w, "desc": 1})
rows["BM25F по полям + описание"] = recall(W, Q, hard30)

pd.DataFrame(rows).T.round(4)

,R@50,R@200,R@1000,R@50 seen,R@50 unseen
заголовок + сырые параметры,0.7159,0.7911,0.8716,0.7279,0.7086
"BM25F по полям, без описания",0.6983,0.7877,0.8686,0.7036,0.6952
BM25F по полям + описание,0.8308,0.8828,0.9214,0.8373,0.8268


Главное число решения: **описание объявления даёт около +0.11 R@50** и поднимает состав пула
(R@1000) с 0.87 до 0.92. Разбор параметров на поля сам по себе почти ничего не даёт — его смысл
в том, что появляются отдельные поля для BM25F.

Обрезать описание не нужно: проверка по длине (`scripts/eval_desc.py`) даёт монотонный рост
200 → 400 → 800 → 1600 → полное, то есть 0.767 → 0.784 → 0.805 → 0.823 → 0.831.

## 4. Гео: приоритет радиуса против мягкого штрафа

Грубый приоритет радиуса игнорирует расстояние внутри радиуса, хотя медиана до позитива 3.9 км,
а 75-й процентиль 10 км. Сравниваем жёсткий приоритет, мягкий штраф `−mu·log(1+dist)` и гибрид.

Здесь важны обе метрики сразу: R@50 отвечает за прямую выдачу, R@1000 — за состав пула, который
потом переупорядочивает ранкер. Лучший вариант по одной метрике не лучший по другой.

In [7]:
variants = {
    "радиус 30 км":               lambda sl: 1000.0 * (dist[sl] < 30),
    "радиус 50 км":               lambda sl: 1000.0 * (dist[sl] < 50),
    "радиус 80 км":               lambda sl: 1000.0 * (dist[sl] < 80),
    "радиус 50 км + штраф 1·log": lambda sl: 1000.0 * (dist[sl] < 50) - 1.0 * log_dist[sl],
    "только штраф 1·log":         lambda sl: -1.0 * log_dist[sl],
    "только штраф 3·log":         lambda sl: -3.0 * log_dist[sl],
}
geo_tab = pd.DataFrame({name: recall(W, Q, fn) for name, fn in variants.items()}).T
geo_tab.round(4)

,R@50,R@200,R@1000,R@50 seen,R@50 unseen
радиус 30 км,0.8308,0.8828,0.9214,0.8373,0.8268
радиус 50 км,0.8382,0.8890,0.9292,0.8504,0.8308
радиус 80 км,0.8374,0.8933,0.9321,0.8549,0.8268
радиус 50 км + штраф 1·log,0.8392,0.8957,0.9369,0.8460,0.8351
только штраф 1·log,0.7843,0.8883,0.9573,0.7836,0.7846
только штраф 3·log,0.8308,0.9018,0.9553,0.8368,0.8273


Развилка: гибрид «радиус 50 км + штраф» даёт лучшую прямую выдачу, а мягкий штраф без радиуса —
лучший состав пула (R@1000 около 0.957 против 0.92). Для генерации кандидатов важнее состав:
то, что не попало в пул, ранкер уже не вернёт.

Отсюда решение — **пул из двух каналов**: половина мест по жёсткому приоритету радиуса, половина
по мягкому штрафу. Ранги в каждом канале уходят в признаки ранкера, пусть решает сам.

## 5. Микрокатегория запроса

86% выборов приходятся на доминирующую микрокатегорию запроса, и запрос «ремонт стиральных машин»
может не иметь общих лемм с объявлением «мастер по стиралкам», но лежать в той же микрокатегории.
Обучаем `P(microcat | запрос)` наивным байесом по леммам — только на обучающей части.

Метрика: доля событий, где истинная микрокатегория попала в top-k предсказаний.

In [8]:
run("scripts/train_microcat.py", "--seed", str(SEED))
pd.read_csv(ROOT / f"results/microcat_seed{SEED}/summary.csv", index_col=0).round(4)

обучающих пар (запрос, microcat): 96982, уникальных запросов 71831, классов 212, 0 с
  {"model": "word_a0.1", "top1": 0.694, "top3": 0.8826666666666667, "top5": 0.92, "top10": 0.9543333333333334, "top20": 0.9733333333333334, "top5_seen": 0.9466666666666667, "top5_unseen": 0.904}
  {"model": "word_a0.01", "top1": 0.685, "top3": 0.878, "top5": 0.9176666666666666, "top10": 0.9536666666666667, "top20": 0.973, "top5_seen": 0.9431111111111111, "top5_unseen": 0.9024}
  {"model": "char_a0.01", "top1": 0.6673333333333333, "top3": 0.8706666666666667, "top5": 0.9123333333333333, "top10": 0.9496666666666667, "top20": 0.972, "top5_seen": 0.9324444444444444, "top5_unseen": 0.9002666666666667}
  {"model": "both_a0.01", "top1": 0.6716666666666666, "top3": 0.8766666666666667, "top5": 0.9176666666666666, "top10": 0.9523333333333334, "top20": 0.9716666666666667, "top5_seen": 0.9404444444444444, "top5_unseen": 0.904}

              top1    top3    top5   top10   top20  top5_seen  top5_unseen
model
word_a0

,top1,top3,top5,top10,top20,top5_seen,top5_unseen
model,,,,,,,
word_a0.1,0.6940,0.8827,0.9200,0.9543,0.9733,0.9467,0.9040
word_a0.01,0.6850,0.8780,0.9177,0.9537,0.9730,0.9431,0.9024
char_a0.01,0.6673,0.8707,0.9123,0.9497,0.9720,0.9324,0.9003
both_a0.01,0.6717,0.8767,0.9177,0.9523,0.9717,0.9404,0.9040


## 6. Пул кандидатов

Собираем top-1200 на событие объединением двух гео-каналов и считаем признаки: BM25F суммарно и
по каждому полю, расстояние и флаги радиусов, совпадение локации, `P(microcat)`, популярность
объявления в обучающей части, отзывы, рейтинг, цену, плотность окрестности.

Главная метрика шага — **потолок пула**: какая доля позитивов вообще в нём оказалась. Выше этого
ранкер не прыгнет.

In [9]:
run("scripts/build_pool.py", "--seed", str(SEED), "--pool", "1200", "--desc-chars", "0", "--geo-mode", "union")

индекс и матрицы полей готовы, 41 с
P(microcat|запрос) подключён
  0/3000 событий, 42 с
  500/3000 событий, 46 с
  1000/3000 событий, 49 с
  1500/3000 событий, 53 с
  2000/3000 событий, 56 с
  2500/3000 событий, 60 с
{
  "seed": 42,
  "pool_size": 1200,
  "desc_chars": 0,
  "radius": 30.0,
  "geo_mode": "union",
  "soft_mu": 1.0,
  "pool_rows_per_event": 1016.4446666666666,
  "pool_rows": 3049334,
  "ceiling_R@pool": 0.9697777777777778,
  "positives_total": 3383,
  "positives_in_pool": 3285,
  "median_rank_of_missed": 601.0,
  "missed_no_overlap_share": 0.10609037328094302,
  "seconds": 63.1
}

[64 c]


## 7. Ранкер

Порядок внутри пула задаётся пока грубым правилом «в радиусе → по BM25F». Ранкер учится взвешивать
все сигналы. LightGBM `lambdarank`, группа = событие.

Две вещи, которые здесь важны:

* **Признаки внутри события.** Абсолютный BM25F плохо сравним между запросами — у длинного запроса
  скоры выше. Нужны относительные: скор, делённый на максимум по событию, ранг, доля от медианы.
* **Вес позитива = 1/число позитивов события.** Метрика усредняется по событиям, поэтому событие
  с тремя позитивами не должно тянуть обучение сильнее события с одним.

Оценка честная: кросс-валидация по событиям, модель никогда не видит событие, которое оценивает.

In [10]:
run("scripts/train_ranker.py", "--seed", str(SEED), "--pool", "1200", "--geo-mode", "union", "--neg-per-event", "200")
pd.read_csv(ROOT / f"results/ranker_seed{SEED}_p1200_union/summary.csv", index_col=0).round(4)

пул 3049334 строк, 32 признаков, 0 с
порядок пула без ранкера:  R@50 = 0.8301 (seen 0.8364 / unseen 0.8263)
потолок пула R@1200 = 0.9698
  lambdarank фолд 0: обучено на 602206 строках, 11 с
  lambdarank фолд 1: обучено на 602176 строках, 21 с
  lambdarank фолд 2: обучено на 602188 строках, 30 с
  -> lambdarank: R@50 = 0.9167 (seen 0.9315 / unseen 0.9078)

важность признаков (последний фолд, top-15):
dist           1547
p_mc           1446
dist_rel       1423
reviews        1411
price          1387
rank_soft      1369
n_in_radius    1362
desc_rel       1269
s_desc         1264
bm25f_rel      1196
bm25f          1190
rating         1154
title_rel      1067
s_title        1007
rank            925
  binary фолд 0: обучено на 602206 строках, 42 с
  binary фолд 1: обучено на 602176 строках, 50 с
  binary фолд 2: обучено на 602188 строках, 59 с
  -> binary: R@50 = 0.9016 (seen 0.9173 / unseen 0.8922)

                       R@50  R@50_seen  R@50_unseen    R@10   R@200
model
lambdarank        

,R@50,R@50_seen,R@50_unseen,R@10,R@200
model,,,,,
lambdarank,0.9167,0.9315,0.9078,0.7987,0.9552
binary,0.9016,0.9173,0.8922,0.7591,0.9485
pool_order_baseline,0.8301,0.8364,0.8263,0.7213,0.8811
pool_ceiling_R@1200,0.9698,0.9739,0.9673,NaN,NaN


Ранкер добавляет примерно +0.09 к порядку пула. Попарная функция потерь заметно лучше бинарной
классификации.

Одна ловушка, на которую легко наступить: если оставить все позитивы, а негативы подвыбрать
из топа пула, то позитив на 700-м месте в обучении есть, а негатива на 700-м нет — и модель
выучивает «большой ранг = позитив», то есть обратный порядок. R@50 падает до 0.04. Негативы
подвыбираются случайно по всему пулу.

### Устойчивость по сидам

Сид управляет и разбиением, и составом unseen-части. Проверяем, что результат не привязан к сиду.

In [11]:
for s in (1, 2):
    if not (val_dir(s) / "val_queries.parquet").exists():
        run("scripts/build_val.py", "--seed", str(s))
        run("scripts/train_microcat.py", "--seed", str(s))
        run("scripts/build_pool.py", "--seed", str(s), "--pool", "1200", "--desc-chars", "0", "--geo-mode", "union")
    run("scripts/train_ranker.py", "--seed", str(s), "--pool", "1200", "--geo-mode", "union", "--neg-per-event", "200")

by_seed = pd.concat({s: pd.read_csv(ROOT / f"results/ranker_seed{s}_p1200_union/summary.csv", index_col=0)["R@50"]
                     for s in (SEED, 1, 2)}, axis=1)
by_seed["среднее"] = by_seed.mean(axis=1)
by_seed.round(4)

{
  "seed": 1,
  "fit_rows": 487455,
  "dropped_rows": 10218,
  "val_events": 3000,
  "seen_share": 0.375,
  "mean_positives": 1.145,
  "positives_unique": 3421,
  "corpus_size": 192334,
  "corpus_seen_in_fit_share": 0.0992,
  "positives_in_corpus": 3421,
  "seconds": 6.3
}

[7 c]


обучающих пар (запрос, microcat): 97063, уникальных запросов 71831, классов 212, 0 с
  {"model": "word_a0.1", "top1": 0.7046666666666667, "top3": 0.8766666666666667, "top5": 0.916, "top10": 0.9453333333333334, "top20": 0.9656666666666667, "top5_seen": 0.9457777777777778, "top5_unseen": 0.8981333333333333}
  {"model": "word_a0.01", "top1": 0.701, "top3": 0.8703333333333333, "top5": 0.916, "top10": 0.946, "top20": 0.9636666666666667, "top5_seen": 0.952, "top5_unseen": 0.8944}
  {"model": "char_a0.01", "top1": 0.6736666666666666, "top3": 0.861, "top5": 0.92, "top10": 0.9523333333333334, "top20": 0.969, "top5_seen": 0.9528888888888889, "top5_unseen": 0.9002666666666667}
  {"model": "both_a0.01", "top1": 0.6893333333333334, "top3": 0.8696666666666667, "top5": 0.9183333333333333, "top10": 0.9526666666666667, "top20": 0.9706666666666667, "top5_seen": 0.9475555555555556, "top5_unseen": 0.9008}

              top1    top3    top5   top10   top20  top5_seen  top5_unseen
model
word_a0.1   0.7047 

индекс и матрицы полей готовы, 40 с
P(microcat|запрос) подключён
  0/3000 событий, 41 с
  500/3000 событий, 45 с
  1000/3000 событий, 48 с
  1500/3000 событий, 51 с
  2000/3000 событий, 54 с
  2500/3000 событий, 58 с
{
  "seed": 1,
  "pool_size": 1200,
  "desc_chars": 0,
  "radius": 30.0,
  "geo_mode": "union",
  "soft_mu": 1.0,
  "pool_rows_per_event": 1013.301,
  "pool_rows": 3039903,
  "ceiling_R@pool": 0.9653796296296295,
  "positives_total": 3434,
  "positives_in_pool": 3319,
  "median_rank_of_missed": 1031.0,
  "missed_no_overlap_share": 0.10984848484848485,
  "seconds": 61.3
}

[62 c]


пул 3039903 строк, 32 признаков, 1 с
порядок пула без ранкера:  R@50 = 0.8209 (seen 0.8229 / unseen 0.8198)
потолок пула R@1200 = 0.9654
  lambdarank фолд 0: обучено на 602249 строках, 11 с
  lambdarank фолд 1: обучено на 602157 строках, 21 с
  lambdarank фолд 2: обучено на 602232 строках, 30 с
  -> lambdarank: R@50 = 0.9178 (seen 0.9337 / unseen 0.9082)

важность признаков (последний фолд, top-15):
dist           1565
bm25f_rel      1442
price          1434
n_in_radius    1367
rank_soft      1367
p_mc           1346
reviews        1341
dist_rel       1331
rating         1274
title_rel      1205
desc_rel       1164
bm25f          1160
s_desc         1080
rank            972
s_title         902
  binary фолд 0: обучено на 602249 строках, 42 с
  binary фолд 1: обучено на 602157 строках, 50 с
  binary фолд 2: обучено на 602232 строках, 59 с
  -> binary: R@50 = 0.8826 (seen 0.9059 / unseen 0.8686)

                       R@50  R@50_seen  R@50_unseen    R@10   R@200
model
lambdarank        

{
  "seed": 2,
  "fit_rows": 486760,
  "dropped_rows": 10913,
  "val_events": 3000,
  "seen_share": 0.375,
  "mean_positives": 1.126,
  "positives_unique": 3356,
  "corpus_size": 192325,
  "corpus_seen_in_fit_share": 0.0992,
  "positives_in_corpus": 3356,
  "seconds": 6.1
}

[7 c]


обучающих пар (запрос, microcat): 97111, уникальных запросов 71831, классов 212, 0 с
  {"model": "word_a0.1", "top1": 0.7113333333333334, "top3": 0.8886666666666667, "top5": 0.9246666666666666, "top10": 0.9586666666666667, "top20": 0.9716666666666667, "top5_seen": 0.952, "top5_unseen": 0.9082666666666667}
  {"model": "word_a0.01", "top1": 0.712, "top3": 0.8863333333333333, "top5": 0.9243333333333333, "top10": 0.9573333333333334, "top20": 0.9726666666666667, "top5_seen": 0.9546666666666667, "top5_unseen": 0.9061333333333333}
  {"model": "char_a0.01", "top1": 0.687, "top3": 0.875, "top5": 0.922, "top10": 0.9543333333333334, "top20": 0.971, "top5_seen": 0.9475555555555556, "top5_unseen": 0.9066666666666666}
  {"model": "both_a0.01", "top1": 0.6943333333333334, "top3": 0.884, "top5": 0.9226666666666666, "top10": 0.9563333333333334, "top20": 0.9706666666666667, "top5_seen": 0.9475555555555556, "top5_unseen": 0.9077333333333333}

              top1    top3    top5   top10   top20  top5_seen 

индекс и матрицы полей готовы, 40 с
P(microcat|запрос) подключён
  0/3000 событий, 41 с
  500/3000 событий, 44 с
  1000/3000 событий, 47 с
  1500/3000 событий, 51 с
  2000/3000 событий, 54 с
  2500/3000 событий, 57 с
{
  "seed": 2,
  "pool_size": 1200,
  "desc_chars": 0,
  "radius": 30.0,
  "geo_mode": "union",
  "soft_mu": 1.0,
  "pool_rows_per_event": 1016.7536666666666,
  "pool_rows": 3050261,
  "ceiling_R@pool": 0.9711833333333333,
  "positives_total": 3377,
  "positives_in_pool": 3278,
  "median_rank_of_missed": 810.0,
  "missed_no_overlap_share": 0.09208103130755065,
  "seconds": 60.3
}

[61 c]


пул 3050261 строк, 32 признаков, 0 с
порядок пула без ранкера:  R@50 = 0.8197 (seen 0.8202 / unseen 0.8195)
потолок пула R@1200 = 0.9712
  lambdarank фолд 0: обучено на 602199 строках, 11 с
  lambdarank фолд 1: обучено на 602189 строках, 20 с
  lambdarank фолд 2: обучено на 602168 строках, 30 с
  -> lambdarank: R@50 = 0.9186 (seen 0.9241 / unseen 0.9153)

важность признаков (последний фолд, top-15):
dist_rel       1597
dist           1542
p_mc           1488
price          1420
rank_soft      1395
n_in_radius    1375
desc_rel       1259
bm25f_rel      1245
reviews        1234
rating         1164
s_desc         1145
bm25f          1107
title_rel      1058
s_title         936
rank            853
  binary фолд 0: обучено на 602199 строках, 41 с
  binary фолд 1: обучено на 602189 строках, 49 с
  binary фолд 2: обучено на 602168 строках, 58 с
  -> binary: R@50 = 0.9027 (seen 0.9101 / unseen 0.8983)

                       R@50  R@50_seen  R@50_unseen    R@10   R@200
model
lambdarank        

,42,1,2,среднее
model,,,,
lambdarank,0.9167,0.9178,0.9186,0.9177
binary,0.9016,0.8826,0.9027,0.8956
pool_order_baseline,0.8301,0.8209,0.8197,0.8236
pool_ceiling_R@1200,0.9698,0.9654,0.9712,0.9688


## 8. Сабмит

Финальный проход по бенчмарку. Отличия от валидации только в источниках: корпус — `benchmark_items`,
запросы — `benchmark_queries`, а центры локаций, популярность и классификатор микрокатегорий
считаются по полному train (валидационных событий там нет, прятать нечего).

Ранкер обучается на всех 3000 валидационных событиях: это настоящие события train с настоящими
метками, а признаки для них считались без них самих — то есть ровно в том режиме, в котором
модель работает на бенчмарке.

Перед записью — проверки формата: две колонки, по строке на каждый `query_id`, до 50 уникальных
`item_id` из корпуса.

In [12]:
run("scripts/predict_ranker.py", "--train-seed", str(SEED), "--pool", "1200",
    "--out", "results/answer_reproduced.csv")

ранкер обучен: 603285 строк, 32 признаков, 3285 позитивов
бенчмарк: 2452 запросов, 189212 объявлений, 10 с
индекс готов, 47 с
запросов без центра локации: 0
признаки из train готовы (212 классов microcat), 48 с
доля запросов бенчмарка, виденных в train: 0.375
  0/2452, 49 с
  500/2452, 55 с
  1000/2452, 61 с
  1500/2452, 66 с
  2000/2452, 72 с

все проверки формата пройдены -> results/answer_reproduced.csv  (76 с)

[77 c]


In [13]:
# Сверка с отправленным файлом: он лежит в репозитории как results/answer.csv
def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

ref, got = ROOT / "results/answer.csv", ROOT / "results/answer_reproduced.csv"
print("отправленный :", md5(ref))
print("полученный   :", md5(got))
assert md5(ref) == md5(got), "файл не совпал с отправленным"
print("\nсовпадает побайтово")

ans = pd.read_csv(got)
print(f"\nстрок {len(ans)}, кандидатов в строке {ans.answer.str.split().str.len().unique()}, "
      f"уникальных объявлений использовано {len({x for a in ans.answer for x in a.split()})}")
ans.head(3)

отправленный : 4a68a07cff83c099a354f14469287b18
полученный   : 4a68a07cff83c099a354f14469287b18

совпадает побайтово

строк 2452, кандидатов в строке [50], уникальных объявлений использовано 82610


,query_id,answer
0,70DfDUpwjxB4lzFd,14d535285758af89 deb58d7ad8c27d06 6fc4db453f87...
1,JTrdTaZJvSiLPkXj,1c6d2079e07f4497 422d3ffdd5bbf626 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,d8fce513e4f000a7 dab52187b4500d9b c5d21c0e8c1c...


## Итог

| система | R@50 на валидации |
|---|---|
| BM25 (заголовок + сырые параметры) + радиус 80 км | 0.690 |
| то же, радиус 30 км | 0.705 |
| BM25F по полям + полное описание | 0.831 |
| + гибридное гео | 0.839 |
| **+ пул из двух каналов и ранкер** | **0.918** |

На лидерборде — **0.8427**.

Что смотрели по ходу: R@50 как целевую метрику, R@200 и R@1000 как состав пула (потолок, выше
которого ранкер не прыгнет), разбивку R@50 на seen/unseen (чтобы прирост не оказался «историей
по запросу»), top-k точность классификатора микрокатегорий, важность признаков ранкера и разброс
итога по трём сидам валидационного сплита.